# 📝 에이전트 품질 과제 LV3(통합)

> 이 단원의 모든 것을 **하나의 작은 시스템**으로 통합합니다. 데이터에서 리포트를 자동 생성하고, 운영에 필요한 관측·견고성을 장착합니다.

## 풀이 방법
1. 맨 위 **준비 셀들**을 먼저 실행하세요(성찰 부품·분석 도구·프롬프트·견고 호출이 제공됩니다).
2. 각 문제는 **여러 단계 셀**로 나뉩니다. 위에서부터 순서대로 채우고 마지막 자가채점으로 확인하세요.
3. 1번 자가채점은 여러분이 만든 `reflect` 를 **한 번 더 직접 부릅니다**(모델 호출 2회). 채점 셀을 여러 번 돌리면 그만큼 호출도 늘어납니다.

- 데이터: `data/gym_members.csv`. 산출물은 `output/` 폴더에 저장합니다.

화이팅!

아래 준비 셀들을 먼저 실행하세요.

In [ ]:
# [제공 코드] OpenAI 키 준비 - 이 셀은 실행만 하세요.
# 14~16일차와 같은 방식입니다: .env 파일에 넣어 둔 OPENAI_API_KEY 를 읽어 옵니다.
import os

from dotenv import load_dotenv

load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우
load_dotenv("../../.env") # 교안 폴더 안의 정답 폴더에서 실행하는 경우

# 키를 먼저 확인합니다. 모델을 만든 뒤에 검사하면 인증 오류가 먼저 나서 이 안내가 묻힙니다.
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(
        "이 노트북은 실제 OpenAI 호출이 필요합니다 - OPENAI_API_KEY 를 찾지 못했습니다.\n"
        "  1) 일차 폴더에서  cp .env.example .env\n"
        "  2) .env 를 열어 본인 키를 채우세요\n"
        "  3) 커널을 재시작한 뒤 이 셀부터 다시 실행하세요")

print("OpenAI 키 확인 완료 - 이제 LangChain 으로 모델을 만들 수 있습니다.")

In [ ]:
# [제공 코드] 오늘 쓸 모델 - 실행만 하세요(LangChain 기본 단원에서 만든 것과 같습니다).
from langchain_openai import ChatOpenAI

# temperature=0 : 같은 질문에 되도록 일정한 답을 받는 설정(수업·채점용).
model = ChatOpenAI(model='gpt-4o-mini', temperature=0)

print('모델 준비 완료:', type(model).__name__)

In [ ]:
# [제공 코드] 성찰 루프 부품 - 지난 강의에서 만든 생성·비평·수정 함수입니다(실행만 하세요).
from pydantic import BaseModel, Field


class Critique(BaseModel):
    score: int = Field(ge=1, le=10, description='1~10 종합 점수')
    issues: list[str] = Field(description='개선점 목록(짧게)')


_GEN_SYSTEM = "너는 데이터 분석 리포트 작성자다. 주어진 수치 요약을 바탕으로 핵심 인사이트를 한국어 세 문장으로 써라."
_CRITIC_SYSTEM = ("너는 깐깐한 리포트 편집자다. 아래 리포트를 평가하라. 구체적 수치 인용·해석의 명확성·실행 제안 유무를 "
                  "기준으로 1~10점을 매기고, 개선점을 항목으로 지적하라.")
_REVISE_SYSTEM = "너는 리포트 작성자다. 아래 [리포트]를 [개선점]을 모두 반영해 다시 써라. 한국어 세 문장을 유지하라."


def generate(summary):
    """수치 요약으로 리포트 초안을 쓴다."""
    r = model.invoke([{'role': 'system', 'content': _GEN_SYSTEM},
                      {'role': 'user', 'content': summary}])
    return r.text


def critique(report):
    """리포트를 구조화된 출력(Critique: 점수·개선점)으로 평가한다."""
    critic_model = model.with_structured_output(Critique)
    return critic_model.invoke([{'role': 'system', 'content': _CRITIC_SYSTEM},
                                {'role': 'user', 'content': report}])


def revise(report, issues):
    """리포트와 개선점을 받아 다시 쓴다."""
    user = f'[리포트]\n{report}\n\n[개선점]\n' + '\n'.join(f'- {x}' for x in issues)
    r = model.invoke([{'role': 'system', 'content': _REVISE_SYSTEM},
                      {'role': 'user', 'content': user}])
    return r.text


In [ ]:
# [제공 코드] 집계 요약을 돌려주는 분석 도구입니다 - 실행만 하세요.
import pandas as pd


def summarize_by(df, group_col, value_col):
    """group_col 별 value_col 평균을 내림차순 문자열로 요약한다(리포트 입력용)."""
    s = df.groupby(group_col)[value_col].mean().sort_values(ascending=False).round(1)
    parts = [f"{k} {v}" for k, v in s.items()]
    return f"{group_col}별 평균 {value_col}: " + ", ".join(parts)


In [ ]:
# [제공 코드] 프롬프트 버전 레지스트리(로컬) - 실행만 하세요.
# 프롬프트를 코드가 아니라 파일(data/prompts.yaml)에서 불러오면, 코드 배포 없이 프롬프트만 교체할 수 있습니다.
# Langfuse 를 쓰면 langfuse.get_prompt(이름, version=번호) 가 똑같은 일을 서버에서 해 줍니다.
from pathlib import Path

import yaml

# 노트북 위치에 따라 data 폴더가 몇 단계 위인지 달라집니다(일차 폴더 / 교안 폴더 / 교안 폴더 안의 정답).
_PROMPTS_PATH = Path("data/prompts.yaml")
if not _PROMPTS_PATH.exists():
    _PROMPTS_PATH = Path("../data/prompts.yaml")
if not _PROMPTS_PATH.exists():
    _PROMPTS_PATH = Path("../../data/prompts.yaml")
PROMPTS = yaml.safe_load(_PROMPTS_PATH.read_text(encoding="utf-8"))


def get_prompt(name, version):
    """이름·버전으로 프롬프트 문자열을 돌려준다(로컬 버전 사전에서)."""
    return PROMPTS[name][version]


In [ ]:
# [제공 코드] 견고한 호출 래퍼(LV2 에서 만든 것) - 실행만 하세요.
def robust_call(call, num_retries=2, fallback=None):
    """call() 을 재시도하고, 실패하면 fallback 을 쓰고, 그래도 안 되면 실패를 보고한다."""
    last_error = ''
    for attempt in range(num_retries + 1):
        try:
            return {'ok': True, 'value': call(), 'attempts': attempt + 1}
        except Exception as error:
            last_error = str(error)
    if fallback is not None:
        try:
            return {'ok': True, 'value': fallback(), 'used_fallback': True}
        except Exception as error:
            last_error = str(error)
    return {'ok': False, 'error': last_error}


In [ ]:
# [제공 코드] 데이터 로드 + 출력 폴더 준비
import os
gym = pd.read_csv('data/gym_members.csv')
os.makedirs('output', exist_ok=True)
print(gym.head())

## 1. 헬스장 월간 인사이트 리포트 자동 생성기
**배경**: 데이터에서 요약을 뽑아 **초안 → 성찰 루프 → 최종 리포트**를 만들고 파일로 저장하는 시스템을 단계별로 완성합니다.

아래 **1단계 → 4단계** 셀을 순서대로 채우세요. 각 단계에 필요한 변수와 목표가 적혀 있습니다.

### 1단계: 데이터 요약 만들기
`summarize_by` 로 **회원권별 월방문횟수**와 **회원권별 재등록여부** 두 요약을 만들어, `' / '` 로 이어 변수 **`monthly_summary`** 에 담으세요.

In [ ]:
# 여기에 코드를 작성하세요

### 2단계: 성찰 루프 만들기
`generate`·`critique`·`revise` 를 묶어 **`reflect(summary, threshold=8, max_iter=3)`** 를 만드세요 (임계 점수 또는 최대 반복에서 멈춤). 반환은 `(리포트, 점수 이력)` 입니다.

> 자가채점이 `reflect(monthly_summary, threshold=1, max_iter=1)` 을 **직접 한 번 더 호출**해 루프가 실제 `generate`·`critique` 를 부르는지 확인합니다. **인자 이름과 기본값을 그대로** 지키세요. 또 루프 안에서는 준비 셀의 **`generate`·`critique`·`revise` 를 이름 그대로** 부르세요(기본 인자로 받아 두면 자가채점이 부품을 가짜로 바꿔 확인할 때 걸립니다).

In [ ]:
# 여기에 코드를 작성하세요

### 3단계: 리포트 생성
`reflect(monthly_summary)` 를 호출해 최종 리포트를 **`final_report`**, 점수 이력을 **`history`** 에 담으세요.

In [ ]:
# 여기에 코드를 작성하세요

### 4단계: 파일로 저장
최종 리포트를 **`output/gym_report.md`** 파일에 저장하세요. 맨 위에 제목 `'# 헬스장 월간 인사이트'` 를 한 줄 넣고, 그 아래 `final_report` 를 씁니다.

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
import os
# 요약은 계산으로 정해진다 - 1단계를 건너뛰었거나 축을 잘못 잡았으면 여기서 걸린다
assert monthly_summary == (summarize_by(gym, '회원권', '월방문횟수') + ' / '
                           + summarize_by(gym, '회원권', '재등록여부'))
assert isinstance(history, list) and 1 <= len(history) <= 3
assert all(isinstance(s, int) and 1 <= s <= 10 for s in history)
assert os.path.exists('output/gym_report.md')
saved = open('output/gym_report.md', encoding='utf-8').read()
assert '헬스장 월간 인사이트' in saved and final_report in saved
# 'x' 한 글자를 파일에 써도 위 검사는 통과합니다 - 리포트다운 분량인지 함께 봅니다.
assert isinstance(final_report, str) and len(final_report.strip()) >= 30, (
    '최종 리포트는 세 문장짜리 글입니다 - 짧은 글자를 손으로 적어 넣으면 여기서 걸립니다')

# 값은 손으로 적어도 위 검사를 통과합니다. 그래서 '진짜' 부품으로 reflect 를 한 번 더 돌려
# 루프가 실제로 굴러가는지 봅니다(임계 1 · 최대 1회 = 생성 1번 + 비평 1번, 모델 호출 두 번).
live_report, live_history = reflect(monthly_summary, threshold=1, max_iter=1)
assert len(live_history) == 1 and isinstance(live_history[0], int) and 1 <= live_history[0] <= 10
assert isinstance(live_report, str) and len(live_report.strip()) >= 30, (
    'reflect 가 실제 generate·critique 를 불러 리포트를 만들어야 합니다')

# 이어서 부품을 잠시 가짜로 바꿔 루프의 순서·횟수를 확인합니다(모델 호출 없음).
real_parts = (generate, critique, revise)
probe_calls = []
probe_scores = iter([5, 9])

def generate(summary):
    probe_calls.append('generate')
    return '초안'

def critique(report):
    probe_calls.append('critique')
    return Critique(score=next(probe_scores), issues=['수치 추가'])

def revise(report, issues):
    probe_calls.append('revise')
    return report + '+수정'

try:
    probe_report, probe_history = reflect('요약')
finally:
    generate, critique, revise = real_parts

assert probe_history == [5, 9], '임계 점수에 닿을 때까지 비평 점수를 모두 모아야 합니다'
assert probe_calls == ['generate', 'critique', 'revise', 'critique']
assert isinstance(probe_report, str) and probe_report.strip()
print('✅ 통과!')

## 2. 운영 대비 점검: 관측·견고성 장착
**배경**: 리포트 생성기를 실제 서비스로 올리기 전, **운영에 필요한 항목**을 한 번에 점검하는 함수를 만듭니다 (관측성 + 견고성 통합).

아래 **1단계 → 3단계**를 순서대로 채우세요.

### 1단계: 관측 콜백 준비
`langfuse.langchain` 의 `CallbackHandler` 로 관측 핸들러를 만들어 리스트 **`handlers`** 에 담고, 관측이 실제로 켜졌는지를 **`observability_on`**(bool)에 담으세요.

- 판정은 `langfuse` 의 **`get_client().auth_check()`** 로 하세요. 핸들러 **개수로 판정하면 안 됩니다**. 키가 없어도 핸들러 객체는 만들어지고(조용히 비활성) 개수는 1이라, 관측이 꺼진 채 켜졌다고 기록됩니다.

In [ ]:
# 여기에 코드를 작성하세요

### 2단계: 견고한 모델 호출 + 토큰 로그
`robust_call` 로 모델 호출을 감싸 **재시도·폴백**을 갖추고, 응답에서 **토큰 수**를 읽어 기록합니다.

- `primary` = `monthly_summary` 로 `generate` 를 호출하는 함수, `fallback` = `lambda: '요약 생성 실패 - 기본 리포트'`.
- `robust_call(primary, num_retries=1, fallback=fallback)` 의 결과를 **`call_result`** 에 담으세요.
- 이어서 토큰 로그용 호출은 **함수 `observed_invoke(chat_model, text)`** 로 만드세요. LV2 5번에서 만든 것과 같은 모양으로, 받은 모델을 **`config={'callbacks': handlers}`** 와 함께 부르고 응답 객체를 돌려줍니다. 1단계에서 만든 관측이 **실제로 쓰이는 자리**가 여기입니다.
- 그 함수로 `observed_invoke(model, monthly_summary)` 를 불러 응답의 `usage_metadata.get('total_tokens', 0)` 을 **`used_tokens`** 에 담으세요.

In [ ]:
# 여기에 코드를 작성하세요

### 3단계: 운영 점검표 만들기
지금까지의 결과를 모아 **`ops_report`** 딕셔너리를 만드세요. 키는 다음과 같습니다.

- `'observability'`: 관측이 켜졌는지(`observability_on`)
- `'call_ok'`: 견고 호출이 성공했는지(`call_result['ok']`)
- `'prompt_version'`: 쓰는 프롬프트 버전 문자열: `'v2'` (v2 가 **수치 재진술·해석·실행 제안**을 요구해 우리가 세운 리포트 품질 기준과 맞기 때문입니다. 레지스트리에서 두 버전을 직접 읽어 비교해 보세요.)
- `'prompt_text'`: **그 버전의 실제 프롬프트 본문**. `get_prompt('insight_writer', 'v2')` 로 꺼내 담으세요 (버전 이름만 적어 두면 나중에 그 버전이 무슨 내용이었는지 알 수 없습니다).
- `'tokens'`: 사용한 토큰(`used_tokens`)

In [ ]:
# 여기에 코드를 작성하세요

In [ ]:
# [자가채점]
for key in ('observability', 'call_ok', 'prompt_version', 'prompt_text', 'tokens'):
    assert key in ops_report
# 관측 여부는 손으로 True 를 적을 수 있다 - 인증을 여기서 다시 확인해 대조한다
assert ops_report['observability'] is True, (
    '관측이 꺼져 있습니다 - .env 의 LANGFUSE_PUBLIC_KEY / LANGFUSE_SECRET_KEY / LANGFUSE_BASE_URL 을 채우고 '
    '커널을 재시작한 뒤 다시 실행하세요(관측성 단원이라 키 없이는 이 문제가 성립하지 않습니다)')
assert ops_report['observability'] == observability_on == get_client().auth_check()
assert ops_report['call_ok'] is True                 # 폴백 덕분에 결국 성공
assert ops_report['call_ok'] == call_result['ok']
assert ops_report['prompt_version'] == 'v2'
# 레지스트리에서 실제로 꺼냈는지 - v1 을 꺼냈거나 손으로 적었으면 여기서 걸린다
assert ops_report['prompt_text'] == get_prompt('insight_writer', 'v2')
assert ops_report['prompt_text'] != get_prompt('insight_writer', 'v1')
# 토큰은 0 을 적어 넣어도 '>= 0' 은 통과한다 - 실제 호출에서 읽은 값과 대조하고 0 보다 큰지 본다
assert isinstance(ops_report['tokens'], int) and ops_report['tokens'] == used_tokens > 0

# 1단계의 handlers 를 정말 호출에 실었는지 - 가짜 모델을 넣어 config 를 들여다본다(모델 호출 없음)
from types import SimpleNamespace

seen = {}


def probe_invoke(text, **kwargs):
    seen['config'] = kwargs.get('config')
    return SimpleNamespace(text='가짜 응답', usage_metadata={'total_tokens': 1})


observed_invoke(SimpleNamespace(invoke=probe_invoke), '점검용 문장')
assert seen.get('config'), 'observed_invoke 안에서 invoke 에 config 를 함께 넘겨야 합니다'
assert seen['config'].get('callbacks') is handlers, (
    "config={'callbacks': handlers} 로 1단계에서 만든 handlers 를 그대로 넘기세요")
print('✅ 통과!')

---
**축하합니다!** 도구를 정의하고(에이전트·도구), 루프를 통제하고(ReAct), 분석을 자동화하고(분석 도구), 이제 **품질(성찰)과 운영(관측·견고성)** 까지 갖췄습니다. 다음 단원에서는 이 에이전트를 사람이 쓰는 **Streamlit 대시보드**로 감쌉니다.